# Section 11: Model Architecture and Training Procedure
Defines, compiles, and trains a 3-class dermoscopic classifier using EfficientNetB0 with staged fine-tuning. Uses the tf.data pipeline from Section 10, class weights, and the frozen split manifests.


In [1]:
%run 01_config.ipynb

import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from IPython.display import display
from augmentation_utils import augment_training_image

PREPROCESS_TARGET_SIZE = IMAGE_SIZE[0]  # 224

def preprocess_image(image_path, target_size=PREPROCESS_TARGET_SIZE, preprocess_fn=None):
    raw_bytes = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(raw_bytes, channels=3)
    shape = tf.shape(image)
    h = tf.cast(shape[0], tf.float32)
    w = tf.cast(shape[1], tf.float32)
    scale = tf.cast(target_size, tf.float32) / tf.minimum(h, w)
    new_h = tf.cast(tf.math.ceil(h * scale), tf.int32)
    new_w = tf.cast(tf.math.ceil(w * scale), tf.int32)
    image = tf.image.resize(image, [new_h, new_w], method="bilinear")
    image = tf.image.resize_with_crop_or_pad(image, target_size, target_size)
    image = tf.cast(image, tf.float32)
    if preprocess_fn is not None:
        image = preprocess_fn(image)
    else:
        image = image / 255.0
    return image

Creating output structure in: D:\SKIN CANCER/pipeline_output
Set basic random seeds to 42.
No GPU detected. Processing on CPU.


## Load Split Manifests & Validate Counts


In [2]:
d_splits = os.path.join(OUTPUT_ROOT, "splits")
d_models = os.path.join(OUTPUT_ROOT, "models")
d_logs = os.path.join(OUTPUT_ROOT, "training_logs")
os.makedirs(d_models, exist_ok=True)
os.makedirs(d_logs, exist_ok=True)

df_train = pd.read_csv(os.path.join(d_splits, "train_manifest_cropped.csv"))
df_val = pd.read_csv(os.path.join(d_splits, "val_manifest_cropped.csv"))
df_test = pd.read_csv(os.path.join(d_splits, "test_manifest_cropped.csv"))
df_train = df_train.sample(frac=1, random_state=42).reset_index(drop=True)

expected = {"train": 14171, "val": 2927, "test": 2957}
errors = []

print("=== SECTION 11 INPUT DATASET VALIDATION ===")
for name, df in [("train", df_train), ("val", df_val), ("test", df_test)]:
    actual = len(df)
    print(f"{name}: {actual} rows (Expected: {expected[name]})")
    if actual != expected[name]:
        errors.append(f"{name} row mismatch: expected {expected[name]}, got {actual}")

if errors:
    raise ValueError(
        "Section 11 must run on the frozen split manifests, but mismatches were found:\n- "
        + "\n- ".join(errors)
    )

print("\nAll split manifest counts match expected frozen state.")


=== SECTION 11 INPUT DATASET VALIDATION ===
train: 14171 rows (Expected: 14171)
val: 2927 rows (Expected: 2927)
test: 2957 rows (Expected: 2957)

All split manifest counts match expected frozen state.


## Compute Class Weights & Build Datasets (Section 10 Contract)


In [3]:
from sklearn.utils.class_weight import compute_class_weight

train_labels = df_train["final_authoritative_label"].values
train_label_indices = np.array([CLASS_TO_INDEX[l] for l in train_labels])

weights = compute_class_weight("balanced", classes=np.array([0, 1, 2]), y=train_label_indices)
CLASS_WEIGHTS = {i: float(w) for i, w in enumerate(weights)}

print("=== CLASS WEIGHTS ===")
for idx, cls in INDEX_TO_CLASS.items():
    print(f"  {cls} (index {idx}): {CLASS_WEIGHTS[idx]:.4f}")


=== CLASS WEIGHTS ===
  NV (index 0): 0.5353
  MEL (index 1): 1.5218
  BCC (index 2): 2.1060


In [4]:
PIPELINE_BATCH_SIZE = BATCH_SIZE  # 32
PIPELINE_SHUFFLE_BUFFER = 4096
PIPELINE_PREFETCH = tf.data.AUTOTUNE
PIPELINE_NUM_PARALLEL = tf.data.AUTOTUNE

def build_dataset(df, is_training=False, batch_size=PIPELINE_BATCH_SIZE):
    paths = df["full_path"].values
    labels = np.array([CLASS_TO_INDEX[l] for l in df["final_authoritative_label"].values])

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if is_training:
        ds = ds.shuffle(PIPELINE_SHUFFLE_BUFFER, seed=RANDOM_SEED, reshuffle_each_iteration=True)

    def load_and_preprocess(path, label):
        image = preprocess_image(path, preprocess_fn=None)
        return image, label

    ds = ds.map(load_and_preprocess, num_parallel_calls=PIPELINE_NUM_PARALLEL)

    if is_training:
        def augment_fn(image, label):
            image = augment_training_image(image)
            return image, label
        ds = ds.map(augment_fn, num_parallel_calls=PIPELINE_NUM_PARALLEL)

    def apply_backbone_preprocess(image, label):
        image = image * 255.0
        image = tf.keras.applications.efficientnet.preprocess_input(image)
        return image, label

    ds = ds.map(apply_backbone_preprocess, num_parallel_calls=PIPELINE_NUM_PARALLEL)

    ds = ds.batch(batch_size, drop_remainder=False)
    ds = ds.prefetch(PIPELINE_PREFETCH)

    return ds


print("Building datasets...")
print("Order: Load -> [0,1] -> Augment (train only) -> EfficientNet preprocess")
ds_train = build_dataset(df_train, is_training=True)
ds_val = build_dataset(df_val, is_training=False)
ds_test = build_dataset(df_test, is_training=False)
print("Datasets built.")

Building datasets...
Order: Load -> [0,1] -> Augment (train only) -> EfficientNet preprocess
Datasets built.


In [5]:
print("=== PRE-DATASET VALUE RANGE CHECK ===")
for images, labels in ds_train.take(1):
    print("Train batch min:", float(tf.reduce_min(images).numpy()))
    print("Train batch max:", float(tf.reduce_max(images).numpy()))
    print("Train batch dtype:", images.dtype)
    print("Train batch shape:", images.shape)

for images, labels in ds_val.take(1):
    print("Val batch min:", float(tf.reduce_min(images).numpy()))
    print("Val batch max:", float(tf.reduce_max(images).numpy()))
    print("Val batch dtype:", images.dtype)
    print("Val batch shape:", images.shape)

=== PRE-DATASET VALUE RANGE CHECK ===
Train batch min: 0.0
Train batch max: 255.0
Train batch dtype: <dtype: 'float32'>
Train batch shape: (32, 224, 224, 3)
Val batch min: 0.0
Val batch max: 255.0
Val batch dtype: <dtype: 'float32'>
Val batch shape: (32, 224, 224, 3)


## Define Model Architecture


In [6]:
NUM_CLASSES = len(CLASS_NAMES)  # 3

def build_model(num_classes=NUM_CLASSES, input_shape=(PREPROCESS_TARGET_SIZE, PREPROCESS_TARGET_SIZE, 3)):
    """
    Build an EfficientNetB0-based classifier for 3-class dermoscopic classification.

    Architecture:
        - EfficientNetB0 backbone (ImageNet pretrained, frozen initially)
        - GlobalAveragePooling2D
        - Dropout(0.3)
        - Dense(128, relu)
        - Dropout(0.3)
        - Dense(num_classes, softmax, float32 for mixed precision safety)
    """
    base_model = tf.keras.applications.EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=input_shape
    )
    base_model.trainable = False  # Freeze for Phase 1

    inputs = layers.Input(shape=input_shape)
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    # Use float32 output for numerical stability with mixed precision
    outputs = layers.Dense(num_classes, activation="softmax", dtype="float32")(x)

    model = keras.Model(inputs, outputs)
    return model, base_model


model, base_model = build_model()
print(f"Model built. Backbone: EfficientNetB0 (frozen)")
print(f"Total params: {model.count_params():,}")
print(f"Trainable params: {sum(tf.keras.backend.count_params(w) for w in model.trainable_weights):,}")
print(f"Non-trainable params: {sum(tf.keras.backend.count_params(w) for w in model.non_trainable_weights):,}")


Model built. Backbone: EfficientNetB0 (frozen)
Total params: 4,213,926
Trainable params: 164,355
Non-trainable params: 4,049,571.0


## Define Training Constants


In [7]:
# Phase 1: Train head only (backbone frozen)
PHASE1_EPOCHS = 10
PHASE1_LR = 1e-3

# Phase 2: Fine-tune top layers of backbone
PHASE2_EPOCHS = 15
PHASE2_LR = 1e-4
PHASE2_UNFREEZE_FROM = 200  # Unfreeze layers from this index onward

# Early stopping
ES_PATIENCE = 5
ES_MONITOR = "val_loss"
ES_RESTORE_BEST = True

# Reduce LR on plateau
RLROP_FACTOR = 0.5
RLROP_PATIENCE = 3
RLROP_MIN_LR = 1e-7

print("=== TRAINING CONSTANTS ===")
print(f"Phase 1: {PHASE1_EPOCHS} epochs, LR={PHASE1_LR}, backbone frozen")
print(f"Phase 2: {PHASE2_EPOCHS} epochs, LR={PHASE2_LR}, unfreeze from layer {PHASE2_UNFREEZE_FROM}")
print(f"Early stopping: patience={ES_PATIENCE}, monitor={ES_MONITOR}")
print(f"Reduce LR: factor={RLROP_FACTOR}, patience={RLROP_PATIENCE}")


=== TRAINING CONSTANTS ===
Phase 1: 10 epochs, LR=0.001, backbone frozen
Phase 2: 15 epochs, LR=0.0001, unfreeze from layer 200
Early stopping: patience=5, monitor=val_loss
Reduce LR: factor=0.5, patience=3


In [8]:
for images, labels in ds_train.take(1):
    preds = model(images, training=False)
    print("pred shape:", preds.shape)
    print("pred min:", float(tf.reduce_min(preds).numpy()))
    print("pred max:", float(tf.reduce_max(preds).numpy()))
    break

pred shape: (32, 3)
pred min: 0.16897723078727722
pred max: 0.5638556480407715


In [9]:
for images, labels in ds_train.take(1):
    preds = model(images, training=False)
    print(preds[0].numpy())
    print("sum:", float(tf.reduce_sum(preds[0]).numpy()))
    break

[0.33984923 0.31454813 0.34560266]
sum: 1.0


## Phase 1: Train Classification Head (Backbone Frozen)


In [12]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=PHASE1_LR),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)
print("=== PRE-TRAIN VALUE RANGE CHECK ===")
for images, labels in ds_train.take(1):
    print("Train batch min:", float(tf.reduce_min(images).numpy()))
    print("Train batch max:", float(tf.reduce_max(images).numpy()))
    print("Train batch dtype:", images.dtype)
    print("Train batch shape:", images.shape)

for images, labels in ds_val.take(1):
    print("Val batch min:", float(tf.reduce_min(images).numpy()))
    print("Val batch max:", float(tf.reduce_max(images).numpy()))
    print("Val batch dtype:", images.dtype)
    print("Val batch shape:", images.shape)

callbacks_phase1 = [
    keras.callbacks.EarlyStopping(
        monitor=ES_MONITOR, patience=ES_PATIENCE,
        restore_best_weights=ES_RESTORE_BEST, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor=ES_MONITOR, factor=RLROP_FACTOR,
        patience=RLROP_PATIENCE, min_lr=RLROP_MIN_LR, verbose=1
    ),
    keras.callbacks.CSVLogger(
        os.path.join(d_logs, "phase1_training_log.csv"), separator=",", append=False
    ),
    keras.callbacks.ModelCheckpoint(
        os.path.join(d_models, "best_model.keras"),
        monitor=ES_MONITOR,
        save_best_only=True,
        verbose=1
    )
]

print("=== PHASE 1: TRAINING HEAD (BACKBONE FROZEN) ===")
history_phase1 = model.fit(
    ds_train,
    validation_data=ds_val,
    epochs=PHASE1_EPOCHS,
    class_weight=CLASS_WEIGHTS,
    callbacks=callbacks_phase1,
    verbose=1
)

print(f"\nPhase 1 complete. Best val_loss: {min(history_phase1.history['val_loss']):.4f}")
print(f"Best val_accuracy: {max(history_phase1.history['val_accuracy']):.4f}")


=== PRE-TRAIN VALUE RANGE CHECK ===
Train batch min: 0.0
Train batch max: 255.0
Train batch dtype: <dtype: 'float32'>
Train batch shape: (32, 224, 224, 3)
Val batch min: 0.0
Val batch max: 255.0
Val batch dtype: <dtype: 'float32'>
Val batch shape: (32, 224, 224, 3)
=== PHASE 1: TRAINING HEAD (BACKBONE FROZEN) ===
Epoch 1/10
443/443 ━━━━━━━━━━━━━━━━━━━━ 0s 260ms/step - accuracy: 0.6996 - loss: 0.6815
Epoch 1: val_loss improved from None to 0.53322, saving model to D:\SKIN CANCER/pipeline_output\models\best_model.keras

Epoch 1: finished saving model to D:\SKIN CANCER/pipeline_output\models\best_model.keras
443/443 ━━━━━━━━━━━━━━━━━━━━ 144s 317ms/step - accuracy: 0.7045 - loss: 0.6659 - val_accuracy: 0.7725 - val_loss: 0.5332 - learning_rate: 0.0010
Epoch 2/10
443/443 ━━━━━━━━━━━━━━━━━━━━ 0s 273ms/step - accuracy: 0.7247 - loss: 0.6269
Epoch 2: val_loss did not improve from 0.53322
443/443 ━━━━━━━━━━━━━━━━━━━━ 146s 328ms/step - accuracy: 0.7340 - loss: 0.6066 - val_accuracy: 0.7581 - val

## Phase 2: Fine-Tune Top Backbone Layers


In [13]:
# Unfreeze backbone from PHASE2_UNFREEZE_FROM onward
base_model.trainable = True
for layer in base_model.layers[:PHASE2_UNFREEZE_FROM]:
    layer.trainable = False

trainable_backbone_layers = sum(1 for l in base_model.layers if l.trainable)
frozen_backbone_layers = sum(1 for l in base_model.layers if not l.trainable)
print(f"Backbone unfrozen from layer {PHASE2_UNFREEZE_FROM}")
print(f"  Trainable backbone layers: {trainable_backbone_layers}")
print(f"  Frozen backbone layers: {frozen_backbone_layers}")
print(f"  Total trainable params: {sum(tf.keras.backend.count_params(w) for w in model.trainable_weights):,}")


Backbone unfrozen from layer 200
  Trainable backbone layers: 38
  Frozen backbone layers: 200
  Total trainable params: 2,215,059


In [14]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=PHASE2_LR),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

callbacks_phase2 = [
    keras.callbacks.EarlyStopping(
        monitor=ES_MONITOR, patience=ES_PATIENCE,
        restore_best_weights=ES_RESTORE_BEST, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor=ES_MONITOR, factor=RLROP_FACTOR,
        patience=RLROP_PATIENCE, min_lr=RLROP_MIN_LR, verbose=1
    ),
    keras.callbacks.CSVLogger(
        os.path.join(d_logs, "phase2_training_log.csv"), separator=",", append=False
    ),
    keras.callbacks.ModelCheckpoint(
        os.path.join(d_models, "best_model.keras"),
        monitor=ES_MONITOR, save_best_only=True, verbose=1
    )
]

print("=== PHASE 2: FINE-TUNING TOP BACKBONE LAYERS ===")
history_phase2 = model.fit(
    ds_train,
    validation_data=ds_val,
    epochs=PHASE2_EPOCHS,
    class_weight=CLASS_WEIGHTS,
    callbacks=callbacks_phase2,
    verbose=1
)

print(f"\nPhase 2 complete. Best val_loss: {min(history_phase2.history['val_loss']):.4f}")
print(f"Best val_accuracy: {max(history_phase2.history['val_accuracy']):.4f}")


=== PHASE 2: FINE-TUNING TOP BACKBONE LAYERS ===
Epoch 1/15
443/443 ━━━━━━━━━━━━━━━━━━━━ 0s 354ms/step - accuracy: 0.7099 - loss: 0.7533
Epoch 1: val_loss improved from None to 0.55522, saving model to D:\SKIN CANCER/pipeline_output\models\best_model.keras

Epoch 1: finished saving model to D:\SKIN CANCER/pipeline_output\models\best_model.keras
443/443 ━━━━━━━━━━━━━━━━━━━━ 189s 414ms/step - accuracy: 0.7328 - loss: 0.6437 - val_accuracy: 0.7629 - val_loss: 0.5552 - learning_rate: 1.0000e-04
Epoch 2/15
443/443 ━━━━━━━━━━━━━━━━━━━━ 0s 357ms/step - accuracy: 0.7719 - loss: 0.5235
Epoch 2: val_loss improved from 0.55522 to 0.51864, saving model to D:\SKIN CANCER/pipeline_output\models\best_model.keras

Epoch 2: finished saving model to D:\SKIN CANCER/pipeline_output\models\best_model.keras
443/443 ━━━━━━━━━━━━━━━━━━━━ 184s 414ms/step - accuracy: 0.7795 - loss: 0.4995 - val_accuracy: 0.7769 - val_loss: 0.5186 - learning_rate: 1.0000e-04
Epoch 3/15
443/443 ━━━━━━━━━━━━━━━━━━━━ 0s 356ms/step 

## Save Final Model


In [15]:
final_model_path = os.path.join(d_models, "final_model.keras")
model.save(final_model_path)
print(f"Final model saved to: {final_model_path}")

saved_model_path = os.path.join(d_models, "final_savedmodel")
model.export(saved_model_path)
print(f"SavedModel format exported to: {saved_model_path}")

Final model saved to: D:\SKIN CANCER/pipeline_output\models\final_model.keras
INFO:tensorflow:Assets written to: D:\SKIN CANCER/pipeline_output\models\final_savedmodel\assets


INFO:tensorflow:Assets written to: D:\SKIN CANCER/pipeline_output\models\final_savedmodel\assets


Saved artifact at 'D:\SKIN CANCER/pipeline_output\models\final_savedmodel'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_238')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  1589818440016: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  1589818438480: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  1589782874832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1589782885584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1589782887120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1589782887888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1589782886928: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1589782887504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1589790050064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1589790050256: TensorSpec(shape=(), dtype=

## Training History Visualization


In [16]:
def plot_training_history(h1, h2, save_path):
    """Plot combined Phase 1 + Phase 2 training curves."""
    acc = h1.history["accuracy"] + h2.history["accuracy"]
    val_acc = h1.history["val_accuracy"] + h2.history["val_accuracy"]
    loss = h1.history["loss"] + h2.history["loss"]
    val_loss = h1.history["val_loss"] + h2.history["val_loss"]

    epochs = range(1, len(acc) + 1)
    phase1_end = len(h1.history["accuracy"])

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(epochs, loss, "b-", label="Train Loss")
    ax1.plot(epochs, val_loss, "r-", label="Val Loss")
    ax1.axvline(x=phase1_end, color="gray", linestyle="--", alpha=0.5, label="Phase 1\u21922")
    ax1.set_title("Loss")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(epochs, acc, "b-", label="Train Accuracy")
    ax2.plot(epochs, val_acc, "r-", label="Val Accuracy")
    ax2.axvline(x=phase1_end, color="gray", linestyle="--", alpha=0.5, label="Phase 1\u21922")
    ax2.set_title("Accuracy")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"Saved: {save_path}")


plot_training_history(
    history_phase1, history_phase2,
    os.path.join(d_logs, "training_curves.png")
)


Saved: D:\SKIN CANCER/pipeline_output\training_logs\training_curves.png


## Quick Test-Set Evaluation (Preliminary — Full Evaluation in Section 12)


In [17]:
print("=== PRELIMINARY TEST EVALUATION ===")
print("Note: Full per-class metrics, confusion matrix, and error analysis are in Section 12.")
print("This is a quick sanity check only.\n")

best_model_path = os.path.join(d_models, "best_model.keras")
if os.path.exists(best_model_path):
    print(f"Loading best model from: {best_model_path}")
    eval_model = keras.models.load_model(best_model_path)
else:
    print("No checkpoint found. Using final model state.")
    eval_model = model

test_loss, test_acc = eval_model.evaluate(ds_test, verbose=1)
print(f"\nTest loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")
print("\nThis is overall accuracy only. Per-class and melanoma-priority analysis follow in Section 12.")


=== PRELIMINARY TEST EVALUATION ===
Note: Full per-class metrics, confusion matrix, and error analysis are in Section 12.
This is a quick sanity check only.

Loading best model from: D:\SKIN CANCER/pipeline_output\models\best_model.keras
93/93 ━━━━━━━━━━━━━━━━━━━━ 25s 256ms/step - accuracy: 0.8028 - loss: 0.4732

Test loss: 0.4732
Test accuracy: 0.8028

This is overall accuracy only. Per-class and melanoma-priority analysis follow in Section 12.


## Save Training Config


In [18]:
training_config = [
    {"parameter": "backbone", "value": "EfficientNetB0"},
    {"parameter": "backbone_weights", "value": "imagenet"},
    {"parameter": "input_size", "value": f"{PREPROCESS_TARGET_SIZE}x{PREPROCESS_TARGET_SIZE}x3"},
    {"parameter": "num_classes", "value": str(NUM_CLASSES)},
    {"parameter": "head_architecture", "value": "GAP -> Dropout(0.3) -> Dense(128,relu) -> Dropout(0.3) -> Dense(3,softmax)"},
    {"parameter": "loss", "value": "SparseCategoricalCrossentropy"},
    {"parameter": "class_weighting", "value": "balanced (sklearn compute_class_weight)"},
    {"parameter": "mixed_precision", "value": str(USE_MIXED_PRECISION)},
    {"parameter": "phase1_epochs", "value": str(PHASE1_EPOCHS)},
    {"parameter": "phase1_lr", "value": str(PHASE1_LR)},
    {"parameter": "phase1_backbone_frozen", "value": "True"},
    {"parameter": "phase2_epochs", "value": str(PHASE2_EPOCHS)},
    {"parameter": "phase2_lr", "value": str(PHASE2_LR)},
    {"parameter": "phase2_unfreeze_from_layer", "value": str(PHASE2_UNFREEZE_FROM)},
    {"parameter": "early_stopping_patience", "value": str(ES_PATIENCE)},
    {"parameter": "reduce_lr_patience", "value": str(RLROP_PATIENCE)},
    {"parameter": "reduce_lr_factor", "value": str(RLROP_FACTOR)},
    {"parameter": "batch_size", "value": str(PIPELINE_BATCH_SIZE)},
    {"parameter": "phase1_actual_epochs", "value": str(len(history_phase1.history['loss']))},
    {"parameter": "phase2_actual_epochs", "value": str(len(history_phase2.history['loss']))},
    {"parameter": "final_test_loss", "value": f"{test_loss:.4f}"},
    {"parameter": "final_test_accuracy", "value": f"{test_acc:.4f}"},
]

df_training_config = pd.DataFrame(training_config)
config_path = os.path.join(d_logs, "training_config.csv")
df_training_config.to_csv(config_path, index=False)
print(f"Training config saved to: {config_path}")
display(df_training_config)


Training config saved to: D:\SKIN CANCER/pipeline_output\training_logs\training_config.csv


,parameter,value
0,backbone,EfficientNetB0
1,backbone_weights,imagenet
2,input_size,224x224x3
3,num_classes,3
4,head_architecture,"GAP -> Dropout(0.3) -> Dense(128,relu) -> Drop..."
5,loss,SparseCategoricalCrossentropy
6,class_weighting,balanced (sklearn compute_class_weight)
7,mixed_precision,False
8,phase1_epochs,10
9,phase1_lr,0.001


## Section 11 Summary


In [19]:
print("=== SECTION 11 FINAL SUMMARY ===")
print(f"Backbone: EfficientNetB0 (ImageNet pretrained)")
print(f"Head: GAP -> Dropout(0.3) -> Dense(128) -> Dropout(0.3) -> Dense(3, softmax)")
print(f"Loss: SparseCategoricalCrossentropy")
print(f"Class weighting: balanced")
print(f"Mixed precision: {USE_MIXED_PRECISION}")
print(f"")
print(f"Phase 1 (head only): {len(history_phase1.history['loss'])} epochs, LR={PHASE1_LR}")
print(f"  Best val_loss: {min(history_phase1.history['val_loss']):.4f}")
print(f"  Best val_accuracy: {max(history_phase1.history['val_accuracy']):.4f}")
print(f"")
print(f"Phase 2 (fine-tune): {len(history_phase2.history['loss'])} epochs, LR={PHASE2_LR}")
print(f"  Best val_loss: {min(history_phase2.history['val_loss']):.4f}")
print(f"  Best val_accuracy: {max(history_phase2.history['val_accuracy']):.4f}")
print(f"")
print(f"Preliminary test accuracy: {test_acc:.4f}")
print(f"Preliminary test loss: {test_loss:.4f}")

print("\nSaved files:")
print("- models/best_model.keras")
print("- models/final_model.keras")
print("- models/final_savedmodel/")
print("- training_logs/phase1_training_log.csv")
print("- training_logs/phase2_training_log.csv")
print("- training_logs/training_curves.png")
print("- training_logs/training_config.csv")

print("\n=== SECTION 11 DOWNSTREAM CONTRACT ===")
print("Section 12 must load best_model.keras for evaluation.")
print("Section 12 must use the same ds_test (Section 10 contract, no augmentation).")
print("Section 12 must compute per-class metrics, confusion matrix, and melanoma-priority analysis.")
print("Do not overclaim clinical validity from internal test accuracy alone.")

print("\nSection 11 completed.")


=== SECTION 11 FINAL SUMMARY ===
Backbone: EfficientNetB0 (ImageNet pretrained)
Head: GAP -> Dropout(0.3) -> Dense(128) -> Dropout(0.3) -> Dense(3, softmax)
Loss: SparseCategoricalCrossentropy
Class weighting: balanced
Mixed precision: False

Phase 1 (head only): 10 epochs, LR=0.001
  Best val_loss: 0.5114
  Best val_accuracy: 0.7800

Phase 2 (fine-tune): 15 epochs, LR=0.0001
  Best val_loss: 0.4685
  Best val_accuracy: 0.8206

Preliminary test accuracy: 0.8028
Preliminary test loss: 0.4732

Saved files:
- models/best_model.keras
- models/final_model.keras
- models/final_savedmodel/
- training_logs/phase1_training_log.csv
- training_logs/phase2_training_log.csv
- training_logs/training_curves.png
- training_logs/training_config.csv

=== SECTION 11 DOWNSTREAM CONTRACT ===
Section 12 must load best_model.keras for evaluation.
Section 12 must use the same ds_test (Section 10 contract, no augmentation).
Section 12 must compute per-class metrics, confusion matrix, and melanoma-priority anal